In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path().resolve().parent))

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_recall_curve, auc

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.preprocessing.build_dataset import process_file

## Load data files

In [ ]:
X1, y1 = process_file(
    Path("../data/raw/chb01/chb01_01.edf"),
    is_seizure_file=False
)

X2, y2 = process_file(
    Path("../data/raw/chb01/chb01_15.edf"),
    is_seizure_file=True,
    seizure_start=1732,
    preictal_duration=600
)

X = np.vstack((X1, X2))
y = np.hstack((y1, y2))

print("Dataset shape:", X.shape)

## Train model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]

threshold = 0.4
y_pred = (y_prob >= threshold).astype(int)

## Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test, y_pred)
disp=ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")

## Precision-Recall Curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

## PR AUC

In [ ]:
pr_auc = auc(recall, precision)
print("PR AUC:", pr_auc)

## Threshold Behaviour

In [ ]:
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision vs Recall vs Threshold")
plt.legend()
plt.show()